In [1]:
import pandas as pd

In [2]:
import pyarrow as pa
from pyarrow import parquet as pq
parquet_file = pq.ParquetFile(r"D:\DIANN-beg\outputs\trial-lib-to-parquet\report-lib.parquet")
df = parquet_file.read(columns=['Modified.Sequence', 'Stripped.Sequence', 'N.Term', 'C.Term', 'RT', 'IM']).to_pandas().drop_duplicates(subset=['Modified.Sequence'])
len(df)

4150763

In [ ]:
df.reset_index(drop=True, inplace=True)

In [5]:
df.rename(columns={'Stripped.Sequence':'sequence', 'RT':'rt', 'IM':'im'}, inplace=True)

In [7]:
df['miss_cleavage'] = df['sequence'].apply(lambda x : max(0, x.count('K') + x.count('R') - 1))

In [8]:
df['is_prot_nterm'] = df['N.Term'].apply(lambda x : 'false' if x == 0 else 'true')
df['is_prot_cterm'] = df['C.Term'].apply(lambda x : 'false' if x == 0 else 'true')

In [9]:
def transform_ptms(ptms_list): #ptms: post translational modifications
    output = []
    for ptm in ptms_list:
        if ptm == ':4':
            output.append('Carbamidomethyl@C')
        elif ptm == ':35':
            output.append('Oxidation@M')
        elif ptm == ':1':
            output.append('Acetyl@Protein_N-term')
    return output

def parse_modified_sequence(modified_sequence):
    pos = 0 # one-based indexing
    seq_len = len(modified_sequence)
    mods = []
    mod_sites = []
    i = 0
    while i < seq_len:
        if modified_sequence[i] == '(':
            i += 7 # this is length of 'UniMod:'
            mod_sites.append(str(pos))
            mod = ''
            while modified_sequence[i] != ')':
                mod += modified_sequence[i]
                i += 1
            mods.append(mod)
            i += 1
        else:
            i += 1
            pos += 1
    if mods == []:
        return '', ''
    return ";".join(transform_ptms(mods)), ";".join(mod_sites)

In [10]:
df[['mods', 'mod_sites']] = pd.DataFrame(list(df['Modified.Sequence'].apply(parse_modified_sequence)))

In [11]:
df['nAA'] = df['sequence'].apply(len)

In [12]:
df.drop(columns=['Modified.Sequence', 'N.Term', 'C.Term'], inplace=True)

In [13]:
df

,sequence,rt,im,miss_cleavage,is_prot_nterm,is_prot_cterm,mods,mod_sites,nAA
0,AAAAAAAAAAAAAAAAGATCLER,69.497040,1.303857,0,false,false,Carbamidomethyl@C,20,23
1,AAAAAAAAAAAAAAAASAGGK,40.703941,1.117346,0,false,false,,,21
2,AAAAAAAAAAAAAAAASAGGKEAASGPNDS,47.367752,1.479664,0,false,true,,,30
3,AAAAAAAAAAAAAAAGAGAGAK,36.775562,1.147410,0,false,false,,,22
4,AAAAAAAAAAAAAAAGAGAGAKQTPADGEASGESEPAK,36.173153,1.258124,1,false,false,,,38
...,...,...,...,...,...,...,...,...,...
4150758,YYYYMWKYISPLMLLSLLIASVVNMGLSPPGYNAWIEDK,195.808655,1.075024,1,false,false,Oxidation@M;Oxidation@M,13;25,39
4150759,YYYYMWKYISPLMLLSLLIASVVNMGLSPPGYNAWIEDK,183.844650,1.071237,1,false,false,Oxidation@M;Oxidation@M;Oxidation@M,5;13;25,39
4150760,YYYYWHLR,45.339214,0.998775,0,false,false,,,8
4150761,YYYYWHLRK,26.811363,1.039129,1,false,false,,,9


In [15]:
df.to_csv('output/' + 'precursor_diann_digest.tsv', sep='\t', index=False)